# DATN -- Malaria Prototype Contrastive Classifier v2
## Pretrained Prototype Init + Prototype Confidence + GradCAM

**Cai tien chinh vs V1:**

| Tinh nang | V1 | V2 |
|---|---|---|
| Prototype init | Random | **Class-mean tu Phase-1 embeddings** |
| Confidence | softmax-max | **proto_ratio, proto_margin, entropy** |
| Training | Single phase (warmup) | **3-phase curriculum** |
| Interpretability | Score bars | **GradCAM heatmap + proto spatial maps** |
| SupCon alpha | Fixed 0.25 | **Ramping 0.05 -> 0.20** |

**Dataset:** annotation files tai `BASE_ANN` folder
**Output:** `/kaggle/working/malaria_proto_v2/`

## 1. Setup & Imports

## 0b. Diagnostic — Verify Setup

Nếu gặp lỗi, chạy cell này TRƯỚC để verify model forward output format.

In [ ]:
import torch
from model import MalariaProtoCLFv2, MalariaProtoCLF

# Test V2 forward returns tuple of 2
model_v2 = MalariaProtoCLFv2('convnext_tiny.in22k_ft_in1k', num_classes=5, pretrained=False)
x = torch.randn(2, 3, 224, 224)
out = model_v2(x)
assert isinstance(out, tuple) and len(out) == 2, f"V2 must return (proj, logits), got {type(out)}"
proj_feats, logits = out
print(f"V2 OK: proj={proj_feats.shape}, logits={logits.shape}")

# Test get_embeddings()
emb_proj, emb_logits = model_v2.get_embeddings(x)
print(f"get_embeddings OK: {emb_proj.shape}, {emb_logits.shape}")

# Test PrototypeHead.get_distances()
dists = model_v2.clf_head.get_distances(proj_feats)
print(f"get_distances OK: {dists.shape} (should be (2, 5))")

# Test V1 alias (backward compat)
model_v1 = MalariaProtoCLF('convnext_tiny.in22k_ft_in1k', num_classes=5, pretrained=False)
proj1, log1 = model_v1(x)
print(f"V1 alias OK: proj={proj1.shape}, logits={log1.shape}")

print("\n[OK] All forward compatibility checks passed!")

In [ ]:
import os, sys
MALARIA_SRC = "/kaggle/working/malaria_proto_clf_src"
if os.path.exists(MALARIA_SRC):
    sys.path.insert(0, MALARIA_SRC)
    print(f"[OK] Source path: {MALARIA_SRC}")
else:
    sys.path.insert(0, "/home/anhlh/Downloads/malaria_proto_clf_src/malaria_clf")
    print("[OK] Using local source path")

from dataset import CLASS_NAMES, NUM_CLASSES, MalariaDataset, get_transforms
from model import MalariaProtoCLFv2, build_model, compute_class_prototypes
from losses import SupConLoss, DynamicFocalLoss
from calibration import TemperatureScaling, compute_ece, PrototypeConfidenceScorer
from train_v2 import TrainerV2Curriculum, TrainConfigV2
print(f"[OK] Class names: {list(CLASS_NAMES.values())}")
print(f"[OK] NUM_CLASSES: {NUM_CLASSES}")

## 2. Dataset Paths

In [ ]:
BASE_ANN = "/kaggle/input/datasets/khanhtq2101/malaria-parasite/final_malaria_full_class_classification_cropped/5 classes - May 2025"
IMG_BASE = "/kaggle/input/datasets/khanhtq2101/malaria-parasite/final_malaria_full_class_classification_cropped"

for fname in ["train_annotation_5classes.txt", "val_annotation_5classes.txt", "test_annotation_5classes.txt"]:
    path = os.path.join(BASE_ANN, fname)
    if os.path.exists(path):
        with open(path) as f:
            lines = [l for l in f.readlines() if l.strip()]
        print(f"OK {fname}: {len(lines)} samples")
    else:
        print(f"FAIL NOT FOUND: {path}")

## 3. Curriculum Training V2 -- 3-Phase Pipeline

**Luong huấn luyện:**
- Phase 1 (15ep): CE Loss -> backbone + FC -> stable embeddings
- Phase 2 (10ep): Prototype init tu P1 embeddings -> prototype head
- Phase 3 (35ep): Joint SupCon + CE (alpha: 0.05 -> 0.20) -> full fine-tune

In [ ]:
cfg = TrainConfigV2()

cfg.BASE_DIR = BASE_ANN
cfg.TRAIN_ANN = os.path.join(BASE_ANN, "train_annotation_5classes.txt")
cfg.VAL_ANN = os.path.join(BASE_ANN, "val_annotation_5classes.txt")
cfg.TEST_ANN = os.path.join(BASE_ANN, "test_annotation_5classes.txt")
cfg.IMG_BASE = IMG_BASE
cfg.OUTPUT_DIR = "/kaggle/working/malaria_proto_v2"

cfg.BACKBONE = "convnext_tiny.in22k_ft_in1k"
cfg.USE_PROTOTYPE = True
cfg.USE_DUAL_HEAD = False
cfg.IMG_SIZE = 224
cfg.DROPOUT = 0.1

cfg.EPOCHS_P1 = 15
cfg.EPOCHS_P2 = 10
cfg.EPOCHS_P3 = 35
cfg.TOTAL_EPOCHS = cfg.EPOCHS_P1 + cfg.EPOCHS_P2 + cfg.EPOCHS_P3

cfg.LR_P1 = 3e-4
cfg.LR_P2 = 2e-4
cfg.LR_P3_BACKBONE = 3e-5
cfg.LR_P3_HEAD = 2e-4
cfg.WEIGHT_DECAY = 1e-4

cfg.SUPCON_TEMP = 0.07
cfg.ALPHA_START = 0.05
cfg.ALPHA_END = 0.20
cfg.CLF_LOSS_P1 = "ce"
cfg.CLF_LOSS_P3 = "focal"

cfg.DO_CALIBRATION = True
cfg.SAVE_BEST_METRIC = "macro_f1"

print(f"Total epochs: {cfg.TOTAL_EPOCHS} (P1={cfg.EPOCHS_P1}, P2={cfg.EPOCHS_P2}, P3={cfg.EPOCHS_P3})")
print(f"Backbone: {cfg.BACKBONE}")
print(f"Output: {cfg.OUTPUT_DIR}")

## 4. Run Training

In [ ]:
import warnings
warnings.filterwarnings("ignore")

trainer = TrainerV2Curriculum(cfg)
model, history = trainer.run()

print(f"\n{'='*60}")
print(f"[Complete] Best macro-F1: {trainer.best_metric:.4f}")
print(f"Checkpoints: {cfg.OUTPUT_DIR}")
print(f"  - phase1_best.pth  - phase2_best.pth")
print(f"  - best_model_v2.pth  - calibrated_model_v2.pth")

## 5. Training Curves

In [ ]:
import matplotlib.pyplot as plt

h = history
p1_end = cfg.EPOCHS_P1
p2_end = p1_end + cfg.EPOCHS_P2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(h["train_loss"], label="Train Loss", color="blue", linewidth=1.5)
axes[0].plot(h["val_loss"], label="Val Loss", color="orange", linewidth=1.5)
axes[0].axvline(p1_end - 1, color="gray", linestyle="--", alpha=0.7, label="P1->P2")
axes[0].axvline(p2_end - 1, color="gray", linestyle=":", alpha=0.7, label="P2->P3")
axes[0].set_title("Loss Curves"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")

axes[1].plot(h["val_macro_f1"], label="Val Macro-F1", color="green", linewidth=2)
axes[1].axvline(p1_end - 1, color="gray", linestyle="--", alpha=0.7, label="P1->P2")
axes[1].axvline(p2_end - 1, color="gray", linestyle=":", alpha=0.7, label="P2->P3")
axes[1].set_title("Validation Macro-F1"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Macro F1")
axes[1].set_ylim(0.7, 1.0)

plt.suptitle(f"Curriculum Training V2 -- {cfg.BACKBONE}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_DIR, "training_curves_v2.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Evaluation on Test Set

In [ ]:
from evaluate import evaluate

CHECKPOINT = os.path.join(cfg.OUTPUT_DIR, "calibrated_model_v2.pth")
if not os.path.exists(CHECKPOINT):
    CHECKPOINT = os.path.join(cfg.OUTPUT_DIR, "best_model_v2.pth")
    print(f"Using: {CHECKPOINT}")

summary, probs, preds, labels = evaluate(
    checkpoint_path=CHECKPOINT,
    test_ann=cfg.TEST_ANN,
    img_base=cfg.IMG_BASE,
    output_dir=os.path.join(cfg.OUTPUT_DIR, "eval_results_v2"),
    batch_size=64,
)

## 7. Prototype-based Confidence Analysis

**New metrics:**
- `proto_margin`: dist_pred - dist_true (>0 = uncertain)
- `proto_ratio`: dist_pred / (dist_pred + dist_second) (0.5 = equidistant)
- `normalized_entropy`: 1 - H(p)/log(C) (0 = certain, 1 = uncertain)
- `margin_confidence`: P_pred - P_second (softmax-based)

**vs softmax-max:**
- Softmax-max: only shows absolute confidence
- Proto_ratio: shows "tug-of-war" between 2 nearest classes
- Proto_margin: shows where embedding really sits vs correct class

In [ ]:
from misclassification_analysis_v2 import analyze_misclassifications

df_all, df_wrong = analyze_misclassifications(
    checkpoint_path=os.path.join(cfg.OUTPUT_DIR, "calibrated_model_v2.pth"),
    test_ann=cfg.TEST_ANN,
    img_base=cfg.IMG_BASE,
    output_dir=os.path.join(cfg.OUTPUT_DIR, "eval_results_v2"),
    batch_size=64,
    max_per_pair=6,
)
print(f"\n{len(df_wrong)} misclassified out of {len(df_all)}")
print(f"Accuracy: {(df_all['correct'].sum()/len(df_all))*100:.2f}%")

## 8. Proto Confidence Interpretation

**How to read proto_confidence_overview.png:**

1. **Proto Margin histogram** (top-left): Correct samples usually have proto_margin < 0
2. **Proto Ratio histogram** (top-middle): ratio ~ 0.5 = equidistant = NEED ATTENTION
3. **Per-class boxplot** (bottom-left): High proto_ratio = model not confident about that class
4. **Confusion heatmap** (bottom-right): Dark red = uncertain = frequently confused class

In [ ]:
print("INTERPRETATION GUIDE:\n" + "="*60)
print("1. Proto Margin (dist_pred - dist_true)")
print("   < 0 -> embedding near CORRECT class -> confident correct")
print("   ~= 0 -> equidistant -> genuinely ambiguous")
print("   > 0 -> embedding near WRONG class -> uncertain / wrong")
print()
print("2. Proto Ratio = dist_pred / (dist_pred + dist_second)")
print("   < 0.3 -> very confident")
print("   0.3-0.5 -> fairly confident")
print("   0.5 -> equidistant -> NEED ATTENTION")
print("   > 0.7 -> model confused")
print("="*60)

correct_pm = df_all[df_all['correct']]['proto_margin'].mean()
wrong_pm = df_all[~df_all['correct']]['proto_margin'].mean()
unc_count = (df_all['proto_ratio'] > 0.6).sum()
print(f"\nKey Stats:")
print(f"  Correct avg proto_margin: {correct_pm:.4f}")
print(f"  Wrong avg proto_margin:   {wrong_pm:.4f}")
print(f"  Uncertain (ratio>0.6):    {unc_count} ({unc_count/len(df_all)*100:.1f}%)")
print(f"  Very uncertain (ratio>0.7): {(df_all['proto_ratio']>0.7).sum()}")

## 9. GradCAM Interpretability Setup

In [ ]:
from gradcam_interpreter import GradCAMInterpreter, PrototypeHeatmapGenerator
import torch
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT = os.path.join(cfg.OUTPUT_DIR, "calibrated_model_v2.pth")

from train_v2 import load_model_v2
model, temperature = load_model_v2(CHECKPOINT, device)
print(f"Model loaded: {cfg.BACKBONE}")

gradcam = GradCAMInterpreter(model, target_layer_name="backbone.stages.3")
proto_hm_gen = PrototypeHeatmapGenerator(model)
print("GradCAM + ProtoHeatmapGenerator initialized")

## 10. GradCAM -- Demo 1 sample per class

In [ ]:
from torch.utils.data import DataLoader

test_tf = get_transforms("val", img_size=cfg.IMG_SIZE)
test_ds = MalariaDataset(cfg.TEST_ANN, cfg.IMG_BASE, transform=test_tf)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

selected = []
seen = set()
for img, label in test_loader:
    if label.item() not in seen:
        seen.add(label.item())
        selected.append((img, label.item()))
        if len(seen) >= NUM_CLASSES:
            break

print(f"Selected {len(selected)} samples (1 per class)")
for img, lbl in selected:
    print(f"  Class {lbl}: {CLASS_NAMES[lbl]}")

In [ ]:
fig, axes = plt.subplots(2, len(selected), figsize=(4*len(selected), 8))

for idx, (img, label) in enumerate(selected):
    img_cuda = img.to(device)
    probs = gradcam.get_prototype_similarities(img_cuda)
    pred = int(np.argmax(probs))
    overlay = gradcam.generate_overlay(img_cuda, class_idx=pred, alpha=0.4)

    axes[0, idx].imshow(overlay)
    axes[0, idx].set_title(f"True: {CLASS_NAMES[label]}\nPred: {CLASS_NAMES[pred]}\nConf: {max(probs):.3f}", fontsize=9)
    axes[0, idx].axis("off")

    ax = axes[1, idx]
    colors = ["#e74c3c","#3498db","#2ecc71","#f39c12","#95a5a6"]
    bars = ax.bar(list(CLASS_NAMES.values()), probs, color=colors, width=0.6)
    ax.set_ylim(0, 1.1)
    ax.set_title("Proto Similarities", fontsize=9)
    ax.tick_params(axis="x", labelsize=7, rotation=15)
    ax.tick_params(axis="y", labelsize=7)
    for bar, p in zip(bars, probs):
        if p > 0.05:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f"{p:.2f}",
                   ha="center", va="bottom", fontsize=7)
    bars[label].set_edgecolor("green"); bars[label].set_linewidth(2)
    bars[pred].set_edgecolor("red"); bars[pred].set_linewidth(2)

plt.suptitle("GradCAM Overlay + Proto Similarities per Class", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_DIR, "eval_results_v2", "gradcam_overview.png"),
            dpi=130, bbox_inches="tight")
plt.show()
print("[Saved] gradcam_overview.png")

## 11. Prototype Spatial Similarity Maps

Each subplot = spatial cosine similarity with each class prototype.
Red = model thinks this region matches the class.
Compare maps of 2 similar classes (e.g. TA vs TJ) to understand confusion.

In [ ]:
img, label = selected[1]  # TJ class (index 1)
img_cuda = img.to(device)

pred_idx = int(np.argmax(gradcam.get_prototype_similarities(img_cuda)))
print(f"True: {CLASS_NAMES[label]}, Pred: {CLASS_NAMES[pred_idx]}")
spatial_maps = proto_hm_gen.generate_spatial_maps(img_cuda)

img_np = img.squeeze().cpu().numpy().transpose(1, 2, 0)
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
img_np = np.clip(img_np * std + mean, 0, 1)

fig = plt.figure(figsize=(15, 4))
fig.suptitle(f"Prototype Spatial Similarity Maps -- True: {CLASS_NAMES[label]}", fontsize=13, fontweight="bold")

for idx, (cls_name, sim_map) in enumerate(spatial_maps.items()):
    ax = fig.add_subplot(1, len(spatial_maps), idx + 1)
    ax.imshow(img_np)
    im = ax.imshow(sim_map, cmap="jet", alpha=0.5, vmin=0, vmax=1)
    ax.set_title(f"Proto: {cls_name}", fontsize=10, fontweight="bold")
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_DIR, "eval_results_v2", "proto_spatial_maps.png"),
            dpi=130, bbox_inches="tight")
plt.show()
print("[Saved] proto_spatial_maps.png")

## 12. Misclassified Samples -- GradCAM + Proto Metrics

For each top confusion pair, visualize:
- Original image
- GradCAM heatmap (what region the model focuses on)
- Probability bar for all 5 classes
- Proto metrics (proto_margin, proto_ratio, margin_confidence)

In [ ]:
import pandas as pd

df_wrong = pd.read_csv(os.path.join(cfg.OUTPUT_DIR, "eval_results_v2", "misclassified_v2.csv"))
pair_counts = df_wrong.groupby(["true_label","pred_label"]).size().sort_values(ascending=False)
print("Top confusion pairs:")
print(pair_counts.head(10))
top_pairs = pair_counts.head(2).index.tolist()
print(f"Analyzing: {top_pairs}")

In [ ]:
for true_lbl, pred_lbl in top_pairs[:1]:
    pair_df = df_wrong[(df_wrong["true_label"]==true_lbl) & (df_wrong["pred_label"]==pred_lbl)]
    print(f"\n[{true_lbl} -> {pred_lbl}]: {len(pair_df)} samples")
    print(f"  Avg proto_margin: {pair_df['proto_margin'].mean():.4f}")
    print(f"  Avg proto_ratio:  {pair_df['proto_ratio'].mean():.4f}")
    print(f"  Avg max_conf:     {pair_df['max_conf'].mean():.4f}")

    samples_to_show = pair_df.head(3)
    n = len(samples_to_show)
    if n == 0:
        continue

    fig = plt.figure(figsize=(5*n, 12))
    fig.suptitle(f"GradCAM: [{true_lbl} -> {pred_lbl}]\n"
                  f"proto_margin={pair_df['proto_margin'].mean():.3f}, "
                  f"proto_ratio={pair_df['proto_ratio'].mean():.3f}",
                  fontsize=12, fontweight="bold")

    for col_idx, (_, row) in enumerate(samples_to_show.iterrows()):
        try:
            img_pil = Image.open(row["path"]).convert("RGB").resize((cfg.IMG_SIZE, cfg.IMG_SIZE))
            img_tensor = test_tf(img_pil).unsqueeze(0).to(device)
        except:
            print(f"  Cannot load: {row['path']}")
            continue

        overlay = gradcam.generate_overlay(img_tensor, class_idx=row["pred_idx"], alpha=0.4)

        ax_img = fig.add_subplot(4, n, col_idx + 1)
        ax_img.imshow(img_pil)
        ax_img.set_title(f"True: {row['true_label']} | Pred: {row['pred_label']}\nconf={row['max_conf']:.3f}", fontsize=8)
        ax_img.axis("off")

        ax_gc = fig.add_subplot(4, n, n + col_idx + 1)
        ax_gc.imshow(overlay)
        ax_gc.set_title(f"GradCAM (pred={CLASS_NAMES[row['pred_idx']]})", fontsize=8)
        ax_gc.axis("off")

        ax_prob = fig.add_subplot(4, n, 2*n + col_idx + 1)
        probs_row = [row[f"prob_{CLASS_NAMES[i]}"] for i in range(NUM_CLASSES)]
        colors = ["#e74c3c","#3498db","#2ecc71","#f39c12","#95a5a6"]
        bars = ax_prob.bar(list(CLASS_NAMES.values()), probs_row, color=colors, width=0.6)
        ax_prob.set_ylim(0, 1.1); ax_prob.tick_params(axis="x", labelsize=6, rotation=15)
        bars[row["true_idx"]].set_edgecolor("green"); bars[row["true_idx"]].set_linewidth(2.5)
        bars[row["pred_idx"]].set_edgecolor("red"); bars[row["pred_idx"]].set_linewidth(2.5)

        ax_txt = fig.add_subplot(4, n, 3*n + col_idx + 1)
        ax_txt.axis("off")
        ax_txt.text(0.1, 0.8, f"proto_margin: {row['proto_margin']:.4f}", fontsize=9, transform=ax_txt.transAxes)
        ax_txt.text(0.1, 0.6, f"proto_ratio:  {row['proto_ratio']:.4f}", fontsize=9, transform=ax_txt.transAxes)
        ax_txt.text(0.1, 0.4, f"margin_conf:  {row['margin_conf']:.4f}", fontsize=9, transform=ax_txt.transAxes)
        ax_txt.text(0.1, 0.2, f"green=true", fontsize=8, color="green", transform=ax_txt.transAxes)
        ax_txt.text(0.1, 0.0, f"red=pred", fontsize=8, color="red", transform=ax_txt.transAxes)

    plt.tight_layout()
    fname = f"gradcam_analysis_{true_lbl}_to_{pred_lbl}.png"
    plt.savefig(os.path.join(cfg.OUTPUT_DIR, "eval_results_v2", fname), dpi=130, bbox_inches="tight")
    plt.show()
    print(f"[Saved] {fname}")

## 13. Summary Report

**Key metrics:**
- `macro_f1`: average F1 across 5 classes
- `parasite_macro_f1`: average F1 for 4 parasite classes (TA, TJ, S, G)
- `ECE`: Expected Calibration Error (< 0.05 is good)
- `correct avg proto_margin`: should be < -0.1 for good model
- `uncertain (ratio>0.6)`: should be < 5% of total

In [ ]:
import json

summary_path = os.path.join(cfg.OUTPUT_DIR, "eval_results_v2", "summary_metrics.json")
if os.path.exists(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
    print("="*60)
    print("EVALUATION SUMMARY")
    print("="*60)
    print(f"  Overall Accuracy: {summary.get('overall_accuracy', 0)*100:.2f}%")
    print(f"  Macro F1:         {summary.get('macro_f1', 0):.4f}")
    print(f"  Weighted F1:     {summary.get('weighted_f1', 0):.4f}")
    print(f"  ECE:              {summary.get('ece', 0):.4f}")
    print(f"  Parasite Macro F1: {summary.get('parasite_macro_f1', 'N/A')}")

conf_path = os.path.join(cfg.OUTPUT_DIR, "eval_results_v2", "confidence_summary.json")
if os.path.exists(conf_path):
    with open(conf_path) as f:
        conf = json.load(f)
    print()
    print("PROTOTYPE CONFIDENCE SUMMARY")
    print("="*60)
    print(f"  Correct avg proto_margin:   {conf['correct_proto_margin_mean']:.4f}")
    print(f"  Wrong avg proto_margin:     {conf['wrong_proto_margin_mean']:.4f}")
    print(f"  Correct avg proto_ratio:    {conf['correct_proto_ratio_mean']:.4f}")
    print(f"  Wrong avg proto_ratio:      {conf['wrong_proto_ratio_mean']:.4f}")
    print(f"  Uncertain (ratio>0.6):     {conf['uncertain_count']} ({conf['uncertain_count']/conf['total_samples']*100:.1f}%)")
    print(f"  Highly confident (ratio<0.3): {conf['highly_confident']} ({conf['highly_confident']/conf['total_samples']*100:.1f}%)")
else:
    print("Run quick_confidence_report() if needed")

print("\n" + "="*60)
print(f"OUTPUT: {os.path.join(cfg.OUTPUT_DIR, 'eval_results_v2')}/")
print("="*60)

## 14. Comparison: V2 Curriculum vs Baseline

Baseline results (ConvNeXtV2 FC finetune):

| Cau hinh | ACC | Macro F1 | TA F1 | TJ F1 | S F1 | G F1 | Parasite Macro F1 |
|---|---|---:|---:|---:|---:|---:|---:|
| CE Loss (baseline) | 99.67% | 0.9121 | 0.7826 | 0.9048 | 0.9355 | 0.9383 | 0.9015 |
| Curriculum V2 | ? | ? | ? | ? | ? | ? | ? |

In [ ]:
report_path = os.path.join(cfg.OUTPUT_DIR, "eval_results_v2", "classification_report.txt")
if os.path.exists(report_path):
    with open(report_path) as f:
        content = f.read()
    print("V2 classification_report.txt:")
    print(content)
else:
    print("Run Cell 6 first to generate classification report")

## 15. Quick Load & Re-evaluate

Uncomment and run if you already have a trained checkpoint.

In [ ]:
# CHECKPOINT = "/kaggle/working/malaria_proto_v2/calibrated_model_v2.pth"
# from misclassification_analysis_v2 import quick_confidence_report
# quick_confidence_report(checkpoint_path=CHECKPOINT, test_ann=cfg.TEST_ANN,
#                         img_base=cfg.IMG_BASE, output_dir="/kaggle/working/eval_results_v2")
print("[INFO] Uncomment cell above to quick-eval an existing checkpoint")